In [ ]:
import torch
import pandas as pd
import json
from tqdm import tqdm
from unsloth import FastLanguageModel

def format_prompt(question, options=None):
    if options is None:
        options = []
        
    is_mcq = bool(options)
    opts_text = "\nOptions:\n" + "\n".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)]) if is_mcq else ""
    user_content = f"{question}{opts_text}"
    
    if is_mcq:
        system_content = r"""You are a world-class mathematician. You must follow these exact rules:
1. Identify the method or theorem that applies.
2. Show every single step; never skip or combine steps.
3. Perform only ONE operation per line.
4. Verify your answer and do a sanity/magnitude check.
5. MATCHING STEP: You MUST explicitly match your final numerical/algebraic answer to the provided options to find the correct letter.
6. BE CONCISE: Stick strictly to mathematical equations. Do not write long paragraphs of text.
7. MULTIPLE BLANKS: If the question contains multiple [ANS] markers, you MUST provide exactly one answer for each marker. Group all of your answers inside a SINGLE \boxed{} tag, separated by commas, in the exact order the [ANS] markers appear.
8. NO LAZY THINKING: You MUST write out your complete, step-by-step mathematical reasoning inside the <think> tags. Do NOT write 'None', do NOT leave it empty, and do NOT skip steps.

Your internal reasoning MUST be enclosed in <think> tags.
After thinking, output ONLY the single letter of your chosen option inside \boxed{}. NEVER box the numerical value."""
    else:
        system_content = r"""You are a world-class mathematician. You must follow these exact rules:
1. Identify the method or theorem that applies.
2. Show every single step; never skip or combine steps.
3. Verify your answer and do a sanity/magnitude check.
4. BE CONCISE: Stick strictly to mathematical equations. Do not write long paragraphs.
5. CRITICAL: NEVER use decimal approximations. Leave all final answers in fully simplified, exact fractional or radical form.
6. STRICT FORMAT: The final box must contain ONLY the mathematical answer. Do not include variable names like "x=" or units.
7. MULTIPLE BLANKS: If the question contains multiple [ANS] markers, you MUST provide exactly one answer for each marker. Group all of your answers inside a SINGLE \boxed{} tag, separated by commas, in the exact order the [ANS] markers appear.
8. NO LAZY THINKING: You MUST write out your complete, step-by-step mathematical reasoning inside the <think> tags. Do NOT write 'None', do NOT leave it empty, and do NOT skip steps.

Your internal reasoning MUST be enclosed in <think> tags.
After thinking, put your final answer inside a SINGLE \boxed{}."""

    return (
        f"<|im_start|>system\n{system_content}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def run_inference(
    model_path="your-hf-username/qwen-math-151b-champion", # 🔴 UPDATE THIS TO YOUR HUGGING FACE REPO
    test_data_path="data/private.jsonl",
    output_csv_path="submission.csv"
):
    MAX_SEQ_LENGTH = 2048

    print(f"Loading Champion Model from Hugging Face: {model_path}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.float16,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model) 

    print(f"Reading test data from {test_data_path}...")
    test_data = []
    with open(test_data_path, "r") as f:
        for line in f:
            test_data.append(json.loads(line))
            
    print(f"Loaded {len(test_data)} test questions.")

    results = []
    print("\n🚀 GENERATING FINAL SUBMISSION 🚀")
    
    for i, row in enumerate(tqdm(test_data, desc="Processing Test Set"), start=1):
        question_id = row.get("id", row.get("problem_id", len(results))) 
        question_text = row["question"]
        options = row.get("options", [])
        
        prompt = format_prompt(question_text, options)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024, 
            use_cache=True,
            temperature=0.1,     
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )
        
        generated_text = tokenizer.batch_decode(
            outputs[:, inputs.input_ids.shape[1]:], 
            skip_special_tokens=True
        )[0]
        
        results.append({
            "id": question_id,
            "response": generated_text.strip()
        })

        if i % 100 == 0:
            pd.DataFrame(results).to_csv(output_csv_path, index=False)

    pd.DataFrame(results).to_csv(output_csv_path, index=False)

    print("\n" + "="*50)
    print(f"✅ SUBMISSION FULLY SAVED: {output_csv_path}")
    print("="*50)

# Execute the pipeline when the notebook is run
if __name__ == "__main__":
    run_inference()